# Neural Collaborative Filtering (NCF) on Amazon Books

End-to-end demo of `NCFEstimator` on the Amazon Reviews 2023 — Books category.

**Why NCF here?** NCF is the foundational deep collaborative-filtering model — it learns
user and item embeddings jointly (no side features), then combines them via a
generalized-MF path and an MLP path (the **NeuMF** variant). Books has a large catalog
(~150k items in our sample) where pure ID-based CF benefits from the deep architecture.

**What this notebook adds:**

1. NCF training (NeuMF variant) on a sampled-softmax-style binary classification task.
2. A `MatrixFactorizationEstimator` baseline trained with the same protocol.
3. Embedding-based two-stage retrieval — `EmbeddingRetriever` reuses the learned NCF user
   and item factors as a retrieval index, then ranks the top-K candidates. This is the
   pattern used in production at scale, and shines on a large catalog like Books.

**Evaluation protocol**: leave-last-positive-out, sampled ranking (1 positive + 100 negatives).

## 1. Imports

In [1]:
import logging
from pathlib import Path

import numpy as np
import pandas as pd

from skrec.dataset.interactions_dataset import InteractionsDataset
from skrec.dataset.items_dataset import ItemsDataset
from skrec.estimator.embedding.matrix_factorization_estimator import MatrixFactorizationEstimator
from skrec.estimator.embedding.ncf_estimator import NCFEstimator
from skrec.recommender.ranking.ranking_recommender import RankingRecommender
from skrec.scorer.universal import UniversalScorer

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(levelname)s %(message)s")

RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("data/ncf")
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Imports OK")

Imports OK


## 2. Download + sample Amazon Books

We pull the McAuley Lab **Amazon Reviews 2023** 5-core Books rating CSV directly from
HuggingFace via `hf_hub_download` (one ~525 MB file, no `datasets` library needed). This
gives us `(user, item, rating, timestamp)` for every 5-core review. We then sample 100k
users with a deterministic seed.

The full Books metadata file (titles, categories, publisher, price) is 14 GB — impractical
for a notebook. These notebooks therefore rely on **interaction-derived features only**:
item popularity, item rating statistics, user behavioural statistics. The recommender
displays book IDs in place of titles.

The cached parquet is **shared across all four Books notebooks** — first run pays the
download cost (~1–3 min); the rest hit the cache instantly.

In [2]:
INTERACTIONS_PARQUET = RAW_DIR / "interactions.parquet"

TARGET_N_USERS = 100_000
SEED = 42

if INTERACTIONS_PARQUET.exists():
    print(f"Cache hit at {INTERACTIONS_PARQUET} — skipping download.")
else:
    try:
        from huggingface_hub import hf_hub_download
    except ImportError:
        print("Installing huggingface_hub...")
        import subprocess
        import sys

        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
        from huggingface_hub import hf_hub_download

    print("Downloading 5core/rating_only/Books.csv (~525 MB)...")
    csv_path = hf_hub_download(
        repo_id="McAuley-Lab/Amazon-Reviews-2023",
        filename="benchmark/5core/rating_only/Books.csv",
        repo_type="dataset",
    )
    print(f"  -> {csv_path}")

    # The 5core rating-only CSV has columns: user_id, parent_asin, rating, timestamp.
    # The full file is ~9.5M rows; loading it all at default pandas dtypes can OOM.
    # Two-pass chunked approach instead:
    #   Pass 1 — stream user_id only, collect unique users, sample.
    #   Pass 2 — stream all 4 columns, keep only sampled users.
    CHUNK = 2_000_000
    print("Pass 1: streaming unique users...")
    unique_users: set[str] = set()
    for chunk in pd.read_csv(csv_path, chunksize=CHUNK, usecols=["user_id"], dtype={"user_id": str}):
        unique_users.update(chunk["user_id"].unique())
    print(f"  total unique users: {len(unique_users):,}")

    rng = np.random.default_rng(SEED)
    sampled_users = set(
        rng.choice(
            np.array(sorted(unique_users)),
            size=min(TARGET_N_USERS, len(unique_users)),
            replace=False,
        )
    )
    print(f"  sampled: {len(sampled_users):,}")

    print("Pass 2: streaming and filtering...")
    parts = []
    for chunk in pd.read_csv(
        csv_path,
        chunksize=CHUNK,
        dtype={"user_id": str, "parent_asin": str, "rating": "float32", "timestamp": "int64"},
    ):
        parts.append(chunk[chunk["user_id"].isin(sampled_users)])
    df = pd.concat(parts, ignore_index=True)
    df = df.rename(columns={"user_id": "USER_ID", "parent_asin": "ITEM_ID", "timestamp": "TIMESTAMP"})
    n_users = df["USER_ID"].nunique()
    n_items = df["ITEM_ID"].nunique()
    print(f"  kept: {len(df):,} interactions across {n_users:,} users, {n_items:,} items")

    df.to_parquet(INTERACTIONS_PARQUET)
    print(f"Saved -> {INTERACTIONS_PARQUET}")

interactions = pd.read_parquet(INTERACTIONS_PARQUET)
print(f"\nLoaded {len(interactions):,} interactions.")
print(f"  users: {interactions['USER_ID'].nunique():,}  items: {interactions['ITEM_ID'].nunique():,}")
interactions.head(3)

Cache hit at data/raw/interactions.parquet — skipping download.

Loaded 1,216,565 interactions.


  users: 100,000  items: 361,673


,USER_ID,ITEM_ID,rating,TIMESTAMP
0,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1446304000,5.0,1441260345000
1,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1564770672,5.0,1441260365000
2,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1442450703,5.0,1523093714024


## 3. Build implicit-feedback Train / Test

NCF is trained on implicit positive signals plus sampled negatives. We:

1. Keep only **positive** interactions (rating ≥ 4) — same convention as SASRec.
2. Per user, hold out the last positive as the test item.
3. Sample 4 random unseen items per training positive as negatives. The estimator
   sees a balanced binary-classification signal.

The catalog is restricted to items that appear in the training positive set so the
embedding tables aren't dominated by unrated long-tail items.

In [3]:
df = interactions.copy()
df = df[df["rating"] >= 4].reset_index(drop=True)
df = df.sort_values(["USER_ID", "TIMESTAMP"]).reset_index(drop=True)

counts = df.groupby("USER_ID").size()
df = df[df["USER_ID"].isin(counts[counts >= 3].index)].reset_index(drop=True)

# Last positive per user → test
last_idx = df.groupby("USER_ID")["TIMESTAMP"].idxmax()
test_df = df.loc[last_idx].reset_index(drop=True)
train_pos = df.drop(index=last_idx).reset_index(drop=True)
test_df = test_df[test_df["USER_ID"].isin(train_pos["USER_ID"].unique())].reset_index(drop=True)

# Restrict items to the training positives (NCF learns embeddings for these)
catalog_items = sorted(train_pos["ITEM_ID"].unique())
items_df = pd.DataFrame({"ITEM_ID": catalog_items})

print(f"Train positives : {len(train_pos):,}")
print(f"Test items      : {len(test_df):,}  (one per user)")
print(f"Users           : {train_pos.USER_ID.nunique():,}")
print(f"Catalog items   : {len(catalog_items):,}")

Train positives : 928,463
Test items      : 96,481  (one per user)
Users           : 96,481
Catalog items   : 319,265


## 4. Sample negatives

For each training positive, sample `n_neg=4` unseen items as negatives with a global
RNG. The result is a long binary-labeled DataFrame ready for `NCFEstimator`.

In [4]:
n_neg = 4
rng = np.random.default_rng(123)
catalog_arr = np.array(catalog_items)

user_seen = train_pos.groupby("USER_ID")["ITEM_ID"].apply(set).to_dict()

neg_rows = []
for u, seen in user_seen.items():
    n_pos_user = len(seen)
    sampled = rng.choice(catalog_arr, size=n_pos_user * n_neg * 2, replace=True)
    sampled = [s for s in sampled if s not in seen][: n_pos_user * n_neg]
    for itm in sampled:
        neg_rows.append((u, itm))

neg_df = pd.DataFrame(neg_rows, columns=["USER_ID", "ITEM_ID"])
neg_df["OUTCOME"] = 0.0

train_pos["OUTCOME"] = 1.0
train_long = (
    pd.concat(
        [train_pos[["USER_ID", "ITEM_ID", "OUTCOME"]], neg_df[["USER_ID", "ITEM_ID", "OUTCOME"]]],
        ignore_index=True,
    )
    .sample(frac=1.0, random_state=42)
    .reset_index(drop=True)
)

print(f"Train (pos+neg) : {len(train_long):,}  positive rate: {train_long.OUTCOME.mean():.1%}")

Train (pos+neg) : 4,642,315  positive rate: 20.0%


## 5. Save CSVs and Build Datasets

In [5]:
train_path = str(DATA_DIR / "train_interactions.csv")
items_path = str(DATA_DIR / "items.csv")

if not Path(train_path).exists():
    train_long.to_csv(train_path, index=False)
if not Path(items_path).exists():
    items_df.to_csv(items_path, index=False)

interactions_ds = InteractionsDataset(data_location=train_path)
items_ds = ItemsDataset(data_location=items_path)
print(f"Train: {len(train_long):,} rows  Items: {len(items_df):,}")

Train: 4,642,315 rows  Items: 319,265


## 6. Train the Matrix Factorization baseline

A pure-NumPy MF baseline gives us a CF reference point that NCF should beat. We use
**SGD** (not the default ALS) because the Books catalog has 360k items, and ALS solves
a `n_factors × n_factors` linear system per user *and* per item per iteration —
prohibitive at this scale. SGD processes interactions one mini-batch at a time, so
training time scales with the number of interactions rather than the catalog size.

In [6]:
mf_estimator = MatrixFactorizationEstimator(
    n_factors=16,
    algorithm="sgd",
    outcome_type="binary",
    epochs=5,
    learning_rate=0.05,
    random_state=42,
)
mf_recommender = RankingRecommender(scorer=UniversalScorer(estimator=mf_estimator))

print("Training MF baseline (SGD)...")
mf_recommender.train(interactions_ds=interactions_ds, items_ds=items_ds)
print("MF done.")

Training MF baseline (SGD)...


2026-04-29 00:49:31,708 - skrec.scorer.base_scorer - WARNING interactions_df contains 1097 duplicate (USER_ID, ITEM_ID) pair(s). This may cause silent wrong results downstream. Deduplicate if unintentional.


2026-04-29 00:49:31,708 skrec.scorer.base_scorer WARNING interactions_df contains 1097 duplicate (USER_ID, ITEM_ID) pair(s). This may cause silent wrong results downstream. Deduplicate if unintentional.


MF done.


## 7. Train NCF (NeuMF variant)

NeuMF combines a Generalized MF arm with an MLP arm. We use **CPU-friendly sizing** here:
`gmf_embedding_dim=16, mlp_embedding_dim=16, mlp_layers=[32, 16, 8], epochs=2,
batch_size=8192`. The Books catalog has ~150k items; even with these modest sizes the
embedding tables hold ~5M params — large enough to demonstrate the architecture without
saturating CPU. Tune dimensions and epochs up for serious benchmarking on GPU.

Runtime ~5–10 min on CPU at this scale.

In [7]:
ncf_estimator = NCFEstimator(
    ncf_type="neumf",
    gmf_embedding_dim=16,
    mlp_embedding_dim=16,
    mlp_layers=[32, 16, 8],
    dropout=0.1,
    learning_rate=1e-3,
    epochs=2,
    batch_size=8192,
    loss_fn_name="bce",
    random_state=42,
    verbose=1,
)
ncf_recommender = RankingRecommender(scorer=UniversalScorer(estimator=ncf_estimator))

print("Training NCF...")
ncf_recommender.train(interactions_ds=interactions_ds, items_ds=items_ds)
print("NCF done.")

Training NCF...


2026-04-29 00:52:56,818 - skrec.scorer.base_scorer - WARNING interactions_df contains 1097 duplicate (USER_ID, ITEM_ID) pair(s). This may cause silent wrong results downstream. Deduplicate if unintentional.


2026-04-29 00:52:56,818 skrec.scorer.base_scorer WARNING interactions_df contains 1097 duplicate (USER_ID, ITEM_ID) pair(s). This may cause silent wrong results downstream. Deduplicate if unintentional.


2026-04-29 00:53:05,459 - skrec.estimator.embedding.base_pytorch_estimator - INFO Epoch [1/2], Loss: 0.5311


2026-04-29 00:53:05,459 skrec.estimator.embedding.base_pytorch_estimator INFO Epoch [1/2], Loss: 0.5311


2026-04-29 00:53:11,607 - skrec.estimator.embedding.base_pytorch_estimator - INFO Epoch [2/2], Loss: 0.5024


2026-04-29 00:53:11,607 skrec.estimator.embedding.base_pytorch_estimator INFO Epoch [2/2], Loss: 0.5024


NCF done.


## 8. Side-by-side evaluation: MF vs. NCF

Books has ~150k items in our positive-only catalog, so we score per-pair (1 positive +
100 negatives per user) on a 2,000-user random sample. Same protocol applied to both MF
and NCF for direct comparability.

In [8]:
TOP_K, N_NEG, N_EVAL_USERS = 10, 100, 2000
catalog_arr = np.array(catalog_items)
gt = test_df.set_index("USER_ID")["ITEM_ID"].to_dict()
eval_test = test_df.sample(n=min(N_EVAL_USERS, len(test_df)), random_state=42).reset_index(drop=True)
print(f"Evaluating {len(eval_test):,} users (sampled).")

user_seen_all = train_pos.groupby("USER_ID")["ITEM_ID"].apply(set).to_dict()
eval_rng = np.random.default_rng(42)

# Build (user, candidate) pairs once — used by both MF and NCF
pair_user, pair_item = [], []
for _, row in eval_test.iterrows():
    u = row["USER_ID"]
    test_item = row["ITEM_ID"]
    seen = user_seen_all.get(u, set())
    unseen = catalog_arr[~np.isin(catalog_arr, list(seen))]
    neg = eval_rng.choice(unseen, size=min(N_NEG, len(unseen)), replace=False)
    pair_user.extend([u] * (1 + len(neg)))
    pair_item.append(test_item)
    pair_item.extend(neg)
pairs = pd.DataFrame({"USER_ID": pair_user, "ITEM_ID": pair_item})

n_users = len(eval_test)
n_per = 1 + N_NEG


def evaluate(name, estimator):
    # Embedding estimators expose predict_proba_with_embeddings
    # (returns shape (n,) for MF or (n, 1) for NCF — ravel to flat).
    print(f"Scoring {len(pairs):,} pairs through {name}...")
    scores = estimator.predict_proba_with_embeddings(pairs[["USER_ID", "ITEM_ID"]]).ravel()
    scores_2d = scores.reshape(n_users, n_per)
    test_scores = scores_2d[:, 0:1]
    ranks = (scores_2d > test_scores).sum(axis=1) + 1
    hit = (ranks <= TOP_K).astype(int)
    ndcg = np.where(ranks <= TOP_K, 1.0 / np.log2(ranks + 1), 0.0)
    print(f"  {name}: HR@{TOP_K}={hit.mean():.4f}  NDCG@{TOP_K}={ndcg.mean():.4f}  (n={n_users:,})")
    return scores


mf_scores = evaluate("MF ", mf_estimator)
ncf_scores = evaluate("NCF", ncf_estimator)

Evaluating 2,000 users (sampled).


Scoring 202,000 pairs through MF ...
  MF : HR@10=0.1160  NDCG@10=0.0520  (n=2,000)
Scoring 202,000 pairs through NCF...
  NCF: HR@10=0.2100  NDCG@10=0.1110  (n=2,000)


## 9. Sample Recommendations from NCF

Top-10 picks for 5 test users from their candidate pool (1 positive + 100 sampled
negatives), ranked by NCF. Full-catalog ranking is impractical at Books scale; this
shows the top picks from each user's candidate set.

> **Two-stage retrieval note**: in production, an `EmbeddingRetriever` would use the
> learned `U @ I.T` factor scores to narrow ~150k items down to a few hundred
> candidates, after which a precise ranker re-scores them. The retriever-based
> `RankingRecommender(scorer=..., retriever=EmbeddingRetriever(...))` pattern is
> demonstrated in [retrieval_two_stage.ipynb](../generic/retrieval_two_stage.ipynb)
> on a small synthetic catalog where the full matmul is cheap to display.

In [9]:
# No title metadata available — show truncated ITEM_ID instead.
eval_users_list = eval_test["USER_ID"].tolist()
for ui in range(5):
    u = eval_users_list[ui]
    user_pair_idx = slice(ui * n_per, (ui + 1) * n_per)
    user_items = pairs.iloc[user_pair_idx]["ITEM_ID"].values
    user_scores = ncf_scores[user_pair_idx]
    order = np.argsort(-user_scores)
    top = user_items[order][:TOP_K]
    test_item = gt.get(u, "?")
    flag = "HIT" if test_item in top else "MISS"
    print(f"\nUser {u}  |  Test: {test_item}  [{flag}]")
    for r, item_id in enumerate(top, 1):
        marker = "  <-- TEST" if item_id == test_item else ""
        print(f"  {r:2}. {item_id}{marker}")


User AHA57DKKJMQTTQMAVQZZNGKTS4JA  |  Test: B07ZNW4NVN  [MISS]
   1. B0058KTL34
   2. 1440241317
   3. 0789741075
   4. B00GMWIGBK
   5. 0449810798
   6. 0974312029
   7. 0800728416
   8. 0345534182
   9. 0070382387
  10. 0440224047

User AGZ5CKEAYQ42F6EGFGGNJVSVTEEQ  |  Test: 1464750432  [MISS]
   1. 061552656X
   2. B004Z1G4OG
   3. 1101966602
   4. 0970043740
   5. 0988503093
   6. 1133283756
   7. 0876058349
   8. 1631363239
   9. 161377205X
  10. 0399162135

User AE7C2DXGPVISWJ3GDNFV6DU4AKIQ  |  Test: 0553378376  [MISS]
   1. 038075911X
   2. 0878422927
   3. 1600591949
   4. 1423141407
   5. 0345334302
   6. 0062083279
   7. 0756610591
   8. 0802127371
   9. 1782795855
  10. 0425245713

User AH3Q4G3LUR6STTSDICY5DNEROUTQ  |  Test: 1947501240  [MISS]
   1. 1576879232
   2. B003UHVTPK
   3. 0310352452
   4. 1557508003
   5. B01HN3Z71I
   6. 1524411515
   7. 162033139X
   8. 0996409548
   9. B005O077KM
  10. B01DPRMM50

User AGHT5IGB7YTWC43MIXYC6EJPFQQQ  |  Test: B00LWDQO90  [HIT]
 